In [ ]:
import numpy as np
import pandas as pd
import os, warnings
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier, VotingClassifier
from sklearn.metrics import accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin
import shap
warnings.filterwarnings("ignore")

In [ ]:
train_path = "/kaggle/input/titanic/train.csv"
test_path = "/kaggle/input/titanic/test.csv"
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
X = train.drop(columns=["Survived"])
y = train["Survived"].astype(int)

In [ ]:
class TitanicFeatureBuilder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        df = X.copy()
        df["Title"] = df["Name"].str.extract(r",\s*([^\.]*)\.")[0].fillna("Unknown")
        df["Title"] = df["Title"].replace({
            "Mlle":"Miss","Ms":"Miss","Mme":"Mrs","Lady":"Royalty","Countess":"Royalty","Sir":"Royalty",
            "Don":"Royalty","Dona":"Royalty","Jonkheer":"Royalty","Capt":"Officer","Col":"Officer",
            "Major":"Officer","Dr":"Officer","Rev":"Officer"
        })
        df["FamilySize"] = df["SibSp"].fillna(0) + df["Parch"].fillna(0) + 1
        df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
        df["CabinDeck"] = df["Cabin"].astype(str).str[0].replace("n","U")
        df["FarePerPerson"] = df["Fare"] / df["FamilySize"]
        df["AgeGroup"] = pd.cut(df["Age"], bins=[-1,12,18,30,45,60,200], labels=["Child","Teen","YoungAdult","Adult","MidAge","Senior"])
        return df

In [ ]:
num_features = ["Age","SibSp","Parch","Fare","FamilySize","FarePerPerson"]
cat_features = ["Pclass","Sex","Embarked","Title","CabinDeck","AgeGroup","IsAlone"]
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")),("encoder", ohe)])
preprocessor = ColumnTransformer(transformers=[("num", numeric_transformer, num_features),("cat", categorical_transformer, cat_features)], remainder="drop")


In [ ]:
candidates = {}
candidates["LogisticRegression"] = LogisticRegression(max_iter=1000, random_state=42)
candidates["RandomForest"] = RandomForestClassifier(n_estimators=800, random_state=42, n_jobs=-1, max_features="sqrt")
candidates["GradientBoosting"] = GradientBoostingClassifier(random_state=42)
try:
    from xgboost import XGBClassifier
    candidates["XGBoost"] = XGBClassifier(n_estimators=700, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.8, reg_lambda=1.0, random_state=42, n_jobs=-1, objective="binary:logistic", eval_metric="logloss")
except Exception:
    pass

In [ ]:
def make_pipeline(model, scale=False):
    steps = [("features", TitanicFeatureBuilder()), ("prep", preprocessor)]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("clf", model))
    return Pipeline(steps)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
for name, model in candidates.items():
    use_scaler = "Logistic" in name
    pipe = make_pipeline(model, scale=use_scaler)
    scores = cross_val_score(pipe, X, y, scoring="accuracy", cv=cv, n_jobs=-1)
    cv_results[name] = (scores.mean(), scores.std())
best_model_name = max(cv_results, key=lambda k: cv_results[k][0])
best_model = candidates[best_model_name]

In [ ]:
final_pipe = make_pipeline(best_model, scale=("Logistic" in best_model_name))
final_pipe.fit(X, y)
preds = final_pipe.predict(test.copy()).astype(int)
submission = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": preds})
submission.to_csv("submission.csv", index=False)

In [ ]:
tuned_results = {}
if "RandomForest" in candidates:
    rf_pipe = make_pipeline(candidates["RandomForest"])
    rf_grid = {
        "clf__n_estimators": [600, 1000],
        "clf__max_depth": [None, 6, 10],
        "clf__min_samples_split": [2, 5],
        "clf__min_samples_leaf": [1, 2],
        "clf__max_features": ["sqrt", "log2"]
    }
    rf_search = GridSearchCV(rf_pipe, rf_grid, scoring="accuracy", cv=5, n_jobs=-1, verbose=0)
    rf_search.fit(X, y)
    tuned_results["RandomForest"] = rf_search
if "XGBoost" in candidates:
    xgb_pipe = make_pipeline(candidates["XGBoost"])
    xgb_grid = {
        "clf__n_estimators": [600, 900],
        "clf__max_depth": [3, 4, 5],
        "clf__learning_rate": [0.03, 0.05],
        "clf__subsample": [0.85, 1.0],
        "clf__colsample_bytree": [0.7, 0.9],
        "clf__reg_lambda": [1.0, 2.0]
    }
    xgb_search = GridSearchCV(xgb_pipe, xgb_grid, scoring="accuracy", cv=5, n_jobs=-1, verbose=0)
    xgb_search.fit(X, y)
    tuned_results["XGBoost"] = xgb_search
best_tuned_name = None
best_tuned_pipe = None
if tuned_results:
    best_tuned_name, best_tuned_pipe = max(tuned_results.items(), key=lambda kv: kv[1].best_score_)
    best_tuned_name = best_tuned_name
    best_tuned_pipe = best_tuned_pipe.best_estimator_
if best_tuned_pipe is not None:
    preds_tuned = best_tuned_pipe.predict(test.copy()).astype(int)
    submission_tuned = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": preds_tuned})
    submission_tuned.to_csv("submission_tuned.csv", index=False)


In [ ]:
def piped(est, scale=False):
    return make_pipeline(est, scale=scale)
ens_parts = []
ens_parts.append(("logreg", piped(LogisticRegression(max_iter=1000, random_state=42), scale=True)))
if "RandomForest" in tuned_results:
    ens_parts.append(("rf", tuned_results["RandomForest"].best_estimator_))
elif "RandomForest" in candidates:
    ens_parts.append(("rf", piped(candidates["RandomForest"])))
if "XGBoost" in tuned_results:
    ens_parts.append(("xgb", tuned_results["XGBoost"].best_estimator_))
elif "XGBoost" in candidates:
    ens_parts.append(("xgb", piped(candidates["XGBoost"])))
ens_parts.append(("gbr", piped(candidates["GradientBoosting"])))
voting = VotingClassifier(estimators=ens_parts, voting="soft", n_jobs=-1)
voting_scores = cross_val_score(voting, X, y, scoring="accuracy", cv=5, n_jobs=-1)
stack = StackingClassifier(estimators=ens_parts, final_estimator=LogisticRegression(max_iter=1000, random_state=42), passthrough=False, n_jobs=-1)
stack_scores = cross_val_score(stack, X, y, scoring="accuracy", cv=5, n_jobs=-1)
best_ensemble = stack if stack_scores.mean() >= voting_scores.mean() else voting
best_ensemble.fit(X, y)
preds_ens = best_ensemble.predict(test.copy()).astype(int)
submission_ensemble = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": preds_ens})
submission_ensemble.to_csv("submission_ensemble.csv", index=False)

In [ ]:
prep_only = Pipeline([("features", TitanicFeatureBuilder()), ("prep", preprocessor)])
Xt = prep_only.fit_transform(X, y)
chosen_est = None
chosen_name = None
if "XGBoost" in tuned_results:
    chosen_est = tuned_results["XGBoost"].best_estimator_.named_steps["clf"]
    chosen_name = "XGBoost"
elif "RandomForest" in tuned_results:
    chosen_est = tuned_results["RandomForest"].best_estimator_.named_steps["clf"]
    chosen_name = "RandomForest"
elif "XGBoost" in candidates:
    chosen_est = candidates["XGBoost"]
    chosen_name = "XGBoost"
elif "RandomForest" in candidates:
    chosen_est = candidates["RandomForest"]
    chosen_name = "RandomForest"
else:
    chosen_est = candidates["GradientBoosting"]
    chosen_name = "GradientBoosting"
if not hasattr(chosen_est, "fit"):
    chosen_est = tuned_results[chosen_name].best_estimator_.named_steps["clf"]
try:
    chosen_est.fit(Xt, y)
except Exception:
    pass
ohe = prep_only.named_steps["prep"].named_transformers_["cat"].named_steps["encoder"]
num_names = list(num_features)
cat_names = list(ohe.get_feature_names_out(cat_features))
feat_names = num_names + cat_names
rng = np.random.default_rng(42)
idx = rng.choice(Xt.shape[0], size=min(300, Xt.shape[0]), replace=False)
Xt_bg = Xt[idx]
try:
    explainer = shap.TreeExplainer(chosen_est)
    shap_values = explainer.shap_values(Xt_bg)
    shap.summary_plot(shap_values if isinstance(shap_values, np.ndarray) else shap_values[1], Xt_bg, feature_names=feat_names, plot_type="bar", show=True)
except Exception:
    pass

In [ ]:
import shap

RANDOM_STATE = 42
prep_only = Pipeline([("features", TitanicFeatureBuilder()), ("prep", preprocessor)])
Xt = prep_only.fit_transform(X, y)

chosen_est = None
if "XGBoost" in tuned_results:
    chosen_est = tuned_results["XGBoost"].best_estimator_.named_steps["clf"]
elif "RandomForest" in tuned_results:
    chosen_est = tuned_results["RandomForest"].best_estimator_.named_steps["clf"]
elif "XGBoost" in candidates:
    chosen_est = candidates["XGBoost"]
elif "RandomForest" in candidates:
    chosen_est = candidates["RandomForest"]
else:
    chosen_est = candidates["GradientBoosting"]

if not hasattr(chosen_est, "fit"):
    chosen_est = tuned_results.get("RandomForest", tuned_results.get("XGBoost")).best_estimator_.named_steps["clf"]

chosen_est.fit(Xt, y)

ohe = prep_only.named_steps["prep"].named_transformers_["cat"].named_steps["encoder"]
num_names = list(num_features)
cat_names = list(ohe.get_feature_names_out(cat_features))
feat_names = num_names + cat_names

n_bg = min(200, Xt.shape[0])
idx_bg = np.random.RandomState(RANDOM_STATE).choice(Xt.shape[0], size=n_bg, replace=False)
Xt_bg = Xt[idx_bg]

try:
    explainer = shap.TreeExplainer(chosen_est)
    shap_values = explainer.shap_values(Xt_bg)
    shap.summary_plot(
        shap_values if isinstance(shap_values, np.ndarray) else shap_values[1],
        Xt_bg,
        feature_names=feat_names,
        plot_type="bar",
        show=True
    )
except Exception as e:
    print("SHAP failed gracefully:", e)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Reuse the best ensemble from Cell 9
y_pred_train = best_ensemble.predict(X)
train_acc = accuracy_score(y, y_pred_train)

print(f"Training Accuracy: {train_acc:.4f}\n")

print("Classification Report:")
print(classification_report(y, y_pred_train, digits=4))

cm = confusion_matrix(y, y_pred_train)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Confusion Matrix – Titanic Ensemble Model")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()